# Lesson 6.2 - Data & Feature Pipelines (toy pipeline demo)

This notebook simulates a production-style batch feature pipeline: raw input, quality checks, feature engineering, and curated output for model training.

## Objectives

- Build a reproducible batch pipeline with explicit stages.
- Add lightweight data-quality checks.
- Create training-ready features with leakage-aware transformations.
- Save curated outputs and pipeline metadata.

## Setup

In [ ]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

RAW_DIR = Path("artifacts/lesson6_2/raw")
CURATED_DIR = Path("artifacts/lesson6_2/curated")
RAW_DIR.mkdir(parents=True, exist_ok=True)
CURATED_DIR.mkdir(parents=True, exist_ok=True)

## Section A - Simulate Raw Batch Data

In [ ]:
n = 2000
base_date = pd.Timestamp("2026-01-01")

raw_df = pd.DataFrame(
    {
        "customer_id": np.arange(1, n + 1),
        "snapshot_date": base_date + pd.to_timedelta(rng.integers(0, 90, size=n), unit="D"),
        "tenure_days": rng.integers(1, 1200, size=n),
        "monthly_spend": np.clip(rng.normal(45, 15, size=n), 2, None),
        "support_tickets_30d": rng.poisson(1.2, size=n),
        "last_login_days_ago": rng.integers(0, 90, size=n),
        "region": rng.choice(["north", "south", "east", "west"], size=n),
    }
)

raw_df["churn_label"] = (
    (raw_df["last_login_days_ago"] > 40).astype(int)
    | (raw_df["support_tickets_30d"] > 2).astype(int)
)

raw_path = RAW_DIR / "customer_snapshot.csv"
raw_df.to_csv(raw_path, index=False)

raw_df.head()

## Section B - Data Quality Checks

In [ ]:
def run_quality_checks(df: pd.DataFrame) -> dict:
    checks = {}
    checks["null_ratio_monthly_spend"] = float(df["monthly_spend"].isna().mean())
    checks["null_ratio_region"] = float(df["region"].isna().mean())
    checks["invalid_tenure_rows"] = int((df["tenure_days"] < 0).sum())
    checks["invalid_login_rows"] = int((df["last_login_days_ago"] < 0).sum())
    checks["duplicate_customer_snapshot"] = int(df.duplicated(subset=["customer_id", "snapshot_date"]).sum())
    checks["row_count"] = int(len(df))
    checks["status"] = "pass"

    if (
        checks["null_ratio_monthly_spend"] > 0.02
        or checks["invalid_tenure_rows"] > 0
        or checks["duplicate_customer_snapshot"] > 0
    ):
        checks["status"] = "fail"

    return checks


quality_report = run_quality_checks(raw_df)
quality_report

## Section C - Feature Engineering Pipeline

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["spend_per_ticket_30d"] = out["monthly_spend"] / (1 + out["support_tickets_30d"])
    out["is_dormant_30d"] = (out["last_login_days_ago"] >= 30).astype(int)
    out["tenure_bucket"] = pd.cut(
        out["tenure_days"],
        bins=[0, 90, 365, 730, 99999],
        labels=["new", "growing", "established", "long_term"],
    )

    out = pd.get_dummies(out, columns=["region", "tenure_bucket"], drop_first=False)
    return out


features_df = build_features(raw_df)
features_df.head()

## Section D - Persist Curated Dataset + Metadata

In [ ]:
curated_path = CURATED_DIR / "customer_features_batch.csv"
meta_path = CURATED_DIR / "pipeline_metadata.json"

features_df.to_csv(curated_path, index=False)

metadata = {
    "run_time_utc": datetime.utcnow().isoformat() + "Z",
    "input_path": str(raw_path),
    "output_path": str(curated_path),
    "row_count": int(len(features_df)),
    "feature_count": int(features_df.shape[1]),
    "quality_report": quality_report,
}
meta_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

metadata

## Optional - Airflow DAG Pattern (Pseudo-code)

This cell prints a minimal DAG skeleton to show how the same stages are orchestrated in production schedulers.

In [ ]:
airflow_dag_example = '''
from airflow import DAG
from airflow.operators.python import PythonOperator

with DAG("daily_feature_pipeline", schedule="0 2 * * *", start_date=..., catchup=False) as dag:
    ingest = PythonOperator(task_id="ingest_raw", python_callable=ingest_raw)
    validate = PythonOperator(task_id="validate_data", python_callable=validate_data)
    transform = PythonOperator(task_id="build_features", python_callable=build_features)
    publish = PythonOperator(task_id="publish_curated", python_callable=publish_data)

    ingest >> validate >> transform >> publish
'''

print(airflow_dag_example)

## Connect to Theory

- Section A maps to ingestion layer.
- Section B enforces data quality contracts.
- Section C defines reusable feature logic.
- Section D persists artifacts and metadata for reproducibility.

In production, the same pattern extends with backfills, lineage, SLA dashboards, and feature store materialization.

## Business Case Studies & Exceptions

### Case 1: Churn prediction with daily batch features

- **Pattern**: run nightly ETL + feature pipeline, score users each morning.
- **Benefit**: predictable costs and simple operations.
- **Exception**: campaign-time interventions may need intraday refreshes.

### Case 2: Real-time recommendation systems

- **Pattern**: combine batch historical features with streaming session features.
- **Benefit**: balances stable historical context with fresh intent signals.
- **Exception**: streaming systems add operational complexity (late events, watermarking, on-call burden).

### Case 3: Feature store adoption decision

- **Rule of thumb**: adopt a full feature store when feature reuse and online/offline consistency pain exceed current pipeline overhead.

## Interview Questions & Answers

1. **What is the role of data pipelines in ML systems?**  
   They deliver validated, timely, and reproducible datasets/features for training and inference.

2. **Batch vs streaming in one line?**  
   Batch is scheduled and simpler; streaming is continuous and lower-latency but harder to operate.

3. **What is point-in-time correctness?**  
   Training features must only use information available at prediction time.

4. **What is training-serving skew?**  
   Mismatch between feature computation in model training and production inference.

5. **Why are data quality checks non-negotiable?**  
   They prevent silent corruption propagating into model decisions.

6. **What belongs in pipeline metadata?**  
   input/output paths, run timestamps, quality checks, schema version, and code version.

7. **When does a team need a feature store?**  
   When many models reuse features and online/offline consistency becomes painful.

8. **What causes schema drift incidents?**  
   Upstream schema changes without contract enforcement or compatibility management.

9. **How do you debug a sudden feature distribution shift?**  
   Trace upstream source changes, transformation logic, and ingestion delays by run ID.

10. **Why keep raw snapshots?**  
   For reproducibility, root-cause analysis, and historical backfills.

11. **What is the first reliability improvement for fragile pipelines?**  
   Add explicit quality checks and fail-fast behavior before modeling improvements.

12. **How do you decide refresh cadence?**  
   Based on business decision latency requirements, data arrival patterns, and operating cost.